In [1]:
print("Spark fonctionne !")

In [2]:
bronze_path = "abfss://healthcare@adlgenstorage.dfs.core.windows.net/Bronze/healthcare_dataset_raw_imperfect.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

In [3]:
display(df.limit(10))

In [5]:
#noumbre total des lignes
nombre_lignes = df.count()
#nombre total des colomnes
nombre_colonnes = len(df.columns)

print("nombre_totale_lignes:", nombre_lignes)
print("nombre_totale_colomnes:", nombre_colonnes)

In [6]:
# Afficher la liste des colonnes du dataset
print(df.columns)

In [7]:
#afficher le schema du dataset
df.printSchema()

In [21]:
# Compter les valeurs NULL réellement présentes dans chaque colonne
from pyspark.sql.functions import col, sum, to_date

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [11]:
#comptons le nombre de lignes
nombre_lignes = df.count()
# nombre de lignes  de lignes unique
nombre_uniques = df.dropDuplicates().count()
# calculons le nombre de doublons
doublons = nombre_lignes - nombre_uniques

print("nombre de lignes:",nombre_lignes)
print("nombre unique:", nombre_uniques)
print("nombre de doublons:",doublons)

In [14]:
# Vérifier les différentes valeurs présentes dans Gender


df.groupBy("Gender").count().show()


In [15]:
# Vérifier les différentes valeurs de Blood Type

df.groupBy("Blood Type").count().show()

In [16]:
# Vérifier les différentes valeurs de Admission Type

df.groupBy("Admission Type").count().show()

In [17]:
# Vérifier les différentes valeurs de Medical Condition

df.groupBy("Medical Condition").count().show()

In [20]:
# Afficher les statistiques des colonnes numériques
# count, moyenne, écart-type, minimum et maximum

df.select(
    "Age",
    "Billing Amount",
    "Room Number"
).describe().show()

In [22]:
#verifications des date incoherent

dates_invalides = df.filter(
    to_date(col("Discharge Date")) <
    to_date(col("Date of Admission"))
)

print("Nombre de dates incohérentes :", dates_invalides.count())

In [23]:
# Afficher quelques lignes ayant des dates incohérentes

dates_invalides.select(
    "Name",
    "Date of Admission",
    "Discharge Date"
).show(10, truncate=False)

In [24]:
# Rechercher les âges potentiellement invalides


ages_invalides = df.filter(
    (col("Age") <= 0) | (col("Age") > 120)
)

print("Nombre d'âges invalides :", ages_invalides.count())

In [25]:
# Vérifier les montants de facturation négatifs

montants_invalides = df.filter(
    col("Billing Amount") < 0
)

print("Nombre de montants négatifs :", montants_invalides.count())